# 00 Setup

**Doel:** Eenmalige (maar idempotente) aanmaak van:
- De Unity Catalog `DEMO`
- De vijf schema's: `STAGING_AZURESTORAGE`, `STAGING_SQLSERVER`, `CONFIG`, `INTEGRATION`, `DATAMART`
- Het External Volume op de Azure Storage External Location
- De control table `DEMO.CONFIG.pipeline_sources` met twee seed-rijen

Dit notebook is volledig idempotent: meerdere keren draaien geeft hetzelfde resultaat
en leidt niet tot fouten of dubbele rijen.

In [ ]:
%sql
-- Widgets — `CREATE WIDGET` registreert ze als gewone dbutils-widgets, dus
-- Workflow `base_parameters` blijft override-en.
CREATE WIDGET TEXT catalog DEFAULT "DEMO";
CREATE WIDGET TEXT parquet_storage_location DEFAULT "";
CREATE WIDGET TEXT catalog_managed_location DEFAULT "";

SELECT
  '${catalog}'                  AS catalog,
  '${parquet_storage_location}' AS parquet_storage_location,
  '${catalog_managed_location}' AS catalog_managed_location;

## Stap 1 — Catalog aanmaken

In [ ]:
%sql
CREATE CATALOG IF NOT EXISTS ${catalog}
  MANAGED LOCATION '${catalog_managed_location}';

USE CATALOG ${catalog};

## Stap 2 — Schema's aanmaken

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS ${catalog}.STAGING_AZURESTORAGE;
CREATE SCHEMA IF NOT EXISTS ${catalog}.STAGING_SQLSERVER;
CREATE SCHEMA IF NOT EXISTS ${catalog}.CONFIG;
CREATE SCHEMA IF NOT EXISTS ${catalog}.INTEGRATION;
CREATE SCHEMA IF NOT EXISTS ${catalog}.DATAMART;

## Stap 3 — External Volume aanmaken

Het volume wijst naar de `parquet/` sub-map van de Azure Storage container die wordt
beheerd door de opgegeven External Location.

In [ ]:
%sql
CREATE EXTERNAL VOLUME IF NOT EXISTS ${catalog}.STAGING_AZURESTORAGE.parquet
  LOCATION '${parquet_storage_location}';

## Stap 4 — Control table aanmaken

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS ${catalog}.CONFIG.pipeline_sources (
  source_system  STRING  NOT NULL COMMENT 'Bronsysteem (bijv. azurestorage)',
  source_path    STRING  NOT NULL COMMENT 'Pad naar de bronfolder (volume-pad)',
  file_pattern   STRING  NOT NULL COMMENT 'Glob filter binnen de folder (per doeltabel)',
  target_schema  STRING  NOT NULL COMMENT 'Doelschema binnen de catalog',
  target_table   STRING  NOT NULL COMMENT 'Doeltabelnaam',
  file_format    STRING  NOT NULL COMMENT 'Bestandstype (bijv. parquet)',
  is_active      BOOLEAN NOT NULL COMMENT 'Aan/uit zonder rij te verwijderen',
  load_type      STRING  NOT NULL COMMENT 'full of incremental (Auto Loader)'
)
USING DELTA
COMMENT 'Control table — stuurt de parquet-ingest pipeline aan';

## Stap 5 — Seed-rijen laden (MERGE — idempotent)

In [ ]:
%sql
MERGE INTO ${catalog}.CONFIG.pipeline_sources AS tgt
USING (
  SELECT
    'azurestorage'                                                       AS source_system,
    '/Volumes/' || LOWER('${catalog}') || '/staging_azurestorage/parquet' AS source_path,
    'ORDER_HEADER*.parquet'                                              AS file_pattern,
    'STAGING_AZURESTORAGE'                                               AS target_schema,
    'STG_ORDER_HEADER'                                                   AS target_table,
    'parquet'                                                            AS file_format,
    true                                                                 AS is_active,
    'full'                                                               AS load_type
  UNION ALL
  SELECT
    'azurestorage',
    '/Volumes/' || LOWER('${catalog}') || '/staging_azurestorage/parquet',
    'ORDER_DETAIL*.parquet',
    'STAGING_AZURESTORAGE',
    'STG_ORDER_DETAIL',
    'parquet',
    true,
    'full'
) AS src
ON  tgt.source_system = src.source_system
AND tgt.target_table  = src.target_table
WHEN NOT MATCHED THEN INSERT *;

## Stap 6 — Validatie & row counts

In [ ]:
%sql
SELECT * FROM ${catalog}.CONFIG.pipeline_sources ORDER BY target_table;

## Resultaat

Setup voltooid. De volgende objecten zijn aangemaakt (of bestonden al):

| Object | Pad |
|--------|-----|
| Catalog | `DEMO` |
| Schema | `DEMO.STAGING_AZURESTORAGE` |
| Schema | `DEMO.STAGING_SQLSERVER` |
| Schema | `DEMO.CONFIG` |
| Schema | `DEMO.INTEGRATION` |
| Schema | `DEMO.DATAMART` |
| External Volume | `DEMO.STAGING_AZURESTORAGE.parquet` |
| Control table | `DEMO.CONFIG.pipeline_sources` (2 rijen) |